# **FOREWORD**

This is a simple Logistic Regression blend kernel that inputs public models, trains a logistic regression model and then submits the result for the leaderboard

# **IMPORTS**

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import rankdata

from gc import collect
from tqdm.auto import tqdm

from sklearn.linear_model import LogisticRegression as LRC
from sklearn.model_selection import *
from sklearn.metrics import *
from sklearn.preprocessing import *
from sklearn.pipeline import make_pipeline, Pipeline

In [ ]:
target = "Will_Buy_EV"

# **PREPROCESSING**

In [ ]:
ytrain = pd.read_csv(
    f"/kaggle/input/competitions/playground-series-s6e9/train.csv",
    index_col = "id"
)[target]

Xtrain = pd.read_parquet(
    f"/kaggle/input/notebooks/ravi20076/playgrounds6e9-public-datacollation-v1/OOF_Preds.parquet"
)

Xtest = pd.read_parquet(
    f"/kaggle/input/notebooks/ravi20076/playgrounds6e9-public-datacollation-v1/Mdl_Preds.parquet"
)

sub_fl = pd.read_csv(
    f"/kaggle/input/competitions/playground-series-s6e9/sample_submission.csv",
    index_col = "id",
)

public_sub = pd.read_csv(
    f"/kaggle/input/notebooks/talhatursun/s6e9-daily-rank-average-ensemble/submission.csv"
)[target].values

print(f"---> Shapes = {Xtrain.shape} {ytrain.shape} {Xtest.shape}")

# **MODEL TRAINING**

We have the requisite data here, let's train our meta-model and blend the predictions

In [ ]:
cv        = StratifiedKFold(5, random_state = 42, shuffle = True)
mdl_preds = 0

for c in [0.001, 0.01, 0.1, 0.5, 1, 3, 5, 7.5, 10]:

    pipe      = make_pipeline(*[MinMaxScaler(), LRC(C = c, random_state = 42, max_iter = 100_000)])
    oof_preds = cross_val_predict(pipe, Xtrain, ytrain, cv = cv, method = "predict_proba")[:, 1]
    score     = roc_auc_score(ytrain, oof_preds)
    print(f"---> Cross validation score = {score:,.8f} | C = {c}")

    pipe.fit(Xtrain, ytrain)
    mdl_preds += pipe.predict_proba(Xtest)[:,1]

# **BLIND BLENDING**

Finally we blind blend with a public kernel having a good score

In [ ]:
sub_fl[target] =  0.20 * (rankdata(mdl_preds) / len(sub_fl)) + 0.80 * (rankdata(public_sub) / len(sub_fl))
sub_fl.to_csv("submission.csv")

print()
!head submission.csv